In [ ]:
🔧 데이터 정제 + Full Fine-tuning 코드

In [ ]:
1. 먼저 문제 데이터 확인

In [ ]:
from datasets import load_dataset
import re

# 데이터셋 로드
dataset = load_dataset('smilegate-ai/kor_unsmile')

# 게임에서 부정적으로 간주할 표현 패턴
negative_patterns = [
    r'빡치', r'빡대가리', r'열받', r'짜증', r'미친', r'씹', r'시발', r'씨발',
    r'ㅅㅂ', r'ㅂㅅ', r'병신', r'멍청', r'바보', r'꺼져', r'닥쳐', r'죽어라',
    r'존나', r'졸라', r'쌍', r'개새끼', r'새끼', r'지랄', r'썅',
]

# 악플/욕설 index = 8, clean index = 9
ABUSE_IDX = 8
CLEAN_IDX = 9

def find_mislabeled(dataset_split, patterns):
    """Clean으로 라벨링되었지만 부정 표현이 포함된 샘플 찾기"""
    mislabeled = []
    
    for idx, item in enumerate(dataset_split):
        text = item['문장']
        labels = item['labels']
        
        # Clean=1 이고 악플/욕설=0 인 경우만 검사
        if labels[CLEAN_IDX] == 1 and labels[ABUSE_IDX] == 0:
            for pattern in patterns:
                if re.search(pattern, text):
                    mislabeled.append({
                        'idx': idx,
                        'text': text,
                        'matched': pattern,
                        'labels': labels
                    })
                    break
    
    return mislabeled

# 문제 데이터 찾기
train_mislabeled = find_mislabeled(dataset['train'], negative_patterns)
valid_mislabeled = find_mislabeled(dataset['valid'], negative_patterns)

print(f"🔍 Train에서 재라벨링 필요한 샘플: {len(train_mislabeled)}건")
print(f"🔍 Valid에서 재라벨링 필요한 샘플: {len(valid_mislabeled)}건")

# 샘플 확인
print("\n📋 재라벨링 필요 샘플 예시 (Train):")
for item in train_mislabeled[:10]:
    print(f"  [{item['matched']}] \"{item['text']}\"")

In [ ]:
2. 데이터 정제 함수

In [ ]:
def relabel_dataset(dataset_split, patterns):
    """
    부정 표현이 포함된 clean 샘플을 악플/욕설로 재라벨링
    """
    corrected_data = {
        '문장': [],
        'labels': [],
    }
    
    relabel_count = 0
    
    for item in dataset_split:
        text = item['문장']
        labels = list(item['labels'])  # 복사본 생성
        
        # Clean=1 이고 악플/욕설=0 인 경우만 재검사
        if labels[CLEAN_IDX] == 1 and labels[ABUSE_IDX] == 0:
            should_relabel = False
            for pattern in patterns:
                if re.search(pattern, text):
                    should_relabel = True
                    break
            
            if should_relabel:
                # 재라벨링: clean → 0, 악플/욕설 → 1
                labels[CLEAN_IDX] = 0
                labels[ABUSE_IDX] = 1
                relabel_count += 1
        
        corrected_data['문장'].append(text)
        corrected_data['labels'].append(labels)
    
    print(f"✅ {relabel_count}건 재라벨링 완료")
    return corrected_data

# 데이터 정제 실행
print("📌 Train 데이터 정제 중...")
train_corrected = relabel_dataset(dataset['train'], negative_patterns)

print("📌 Valid 데이터 정제 중...")
valid_corrected = relabel_dataset(dataset['valid'], negative_patterns)

In [ ]:
3. 정제된 데이터셋 생성

In [ ]:
from datasets import Dataset, DatasetDict

# 정제된 Dataset 생성
corrected_dataset = DatasetDict({
    'train': Dataset.from_dict(train_corrected),
    'valid': Dataset.from_dict(valid_corrected),
})

print(f"\n📊 정제된 데이터셋:")
print(f"  Train: {len(corrected_dataset['train'])}건")
print(f"  Valid: {len(corrected_dataset['valid'])}건")

# 정제 결과 확인 (악플/욕설 라벨 분포)
def count_abuse_labels(ds):
    count = sum(1 for item in ds if item['labels'][ABUSE_IDX] == 1)
    return count

print(f"\n🏷️ 악플/욕설 라벨 수:")
print(f"  Train (원본): {count_abuse_labels(dataset['train'])} → (정제): {count_abuse_labels(corrected_dataset['train'])}")
print(f"  Valid (원본): {count_abuse_labels(dataset['valid'])} → (정제): {count_abuse_labels(corrected_dataset['valid'])}")

In [ ]:
4. 이제 Full Fine-tuning 진행

In [ ]:
from transformers import (
    AutoTokenizer, 
    BertForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
import torch
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score

# 모델 & 토크나이저
model_name = 'smilegate-ai/kor_unsmile'
tokenizer = AutoTokenizer.from_pretrained(model_name)

unsmile_labels = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
num_labels = len(unsmile_labels)

model_full = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# 전처리
def preprocess_function(examples):
    # 1. 토큰화 + 숫자 인코딩 (한 번에!)
    tokenized = tokenizer(
        examples["문장"],
        truncation=True, # 128자 초과시 자름
        padding=False, # 나중에 배치 단위로 패딩
        max_length=128
    )
    tokenized["labels"] = [list(map(float, label)) for label in examples["labels"]]
    return tokenized

# ⚠️ 정제된 데이터셋 사용!
tokenized_dataset = corrected_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=corrected_dataset["train"].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 평가 함수
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = (torch.sigmoid(torch.tensor(predictions)) > 0.5).numpy().astype(int)
    labels = labels.astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    abuse_precision, abuse_recall, abuse_f1, _ = precision_recall_fscore_support(
        labels[:, 8], preds[:, 8], average='binary', zero_division=0
    )
    lrap = label_ranking_average_precision_score(labels, predictions)
    
    return {
        'f1': f1, 'precision': precision, 'recall': recall, 'lrap': lrap,
        'abuse_f1': abuse_f1, 'abuse_recall': abuse_recall, 'abuse_precision': abuse_precision,
    }

# 학습 설정
training_args = TrainingArguments(
    output_dir="./full_finetuned_corrected",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="abuse_recall",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model_full,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 🚀 학습 시작
trainer.train()

In [ ]:
5. 결과 확인 및 저장

In [ ]:
# 평가
eval_results = trainer.evaluate()
print("\n" + "="*50)
print("📊 Full Fine-tuning (정제 데이터) 결과")
print("="*50)
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# 저장
trainer.save_model("./full_finetuned_corrected/best")
tokenizer.save_pretrained("./full_finetuned_corrected/best")
print("\n✅ 모델 저장 완료!")

In [ ]:
📋 재라벨링할 표현 패턴 (확장 가능)

In [ ]:
# 게임 상황에서 부정적으로 간주할 표현들
negative_patterns = [
    # 욕설/비속어
    r'빡치', r'빡대가리', r'빡빡', 
    r'열받', r'짜증', 
    r'씹', r'시발', r'씨발', r'ㅅㅂ', r'ㅂㅅ', r'썅',
    r'병신', r'ㅄ', r'멍청', 
    r'미친', r'ㅁㅊ',
    r'존나', r'졸라', r'ㅈㄴ',
    r'새끼', r'개새끼', r'색끼',
    r'지랄',
    
    # 공격적 표현
    r'꺼져', r'닥쳐', r'죽어라',
    
    # 추가로 게임에서 자주 쓰이는 부정 표현
    r'노답', r'쓰레기', r'트롤', r'핵쟁이', r'버스', 
]

In [ ]:
🎯 요약
단계	설명
1. 문제 발견	"빡친다" 등 게임 욕설이 clean으로 라벨링된 샘플 탐지
2. 재라벨링	clean → 악플/욕설로 수정
3. 학습	정제된 데이터로 Full Fine-tuning
4. 효과	abuse_recall 향상 기대!